# AI工学101 — 第34回

## 次元削減の続き：PCAからt-SNE・UMAPへ

### 高次元データを「人間が見える形」にする

よしレベル、今日は**高次元空間の可視化**だ。

第28回ではPCAをやった。

PCAは、

```text
高次元データ
↓
線形変換
↓
分散をよく保つ方向へ圧縮
```

する方法だった。

今日はさらに、

```text
PCA
↓
t-SNE
↓
UMAP
```

という流れを見ていく。

ただし今日の主役は、

> **「きれいな図を作ること」ではなく、可視化結果をどう解釈するか**

だ。

クラスタリングと同じく、

```text
2次元で見えた構造
＝
データの絶対的な真実
```

ではない。

むしろ、

> **高次元空間の情報を、ある方法で2次元へ翻訳した結果**

として扱う。

---

# 🎯 今日のゴール

今日は次のことを身につける。

* PCAと非線形次元削減の違いを説明できる
* t-SNEの基本的な考え方を理解する
* UMAPの基本的な考え方を理解する
* 高次元データを2次元に可視化できる
* `perplexity` や `n_neighbors` の意味をざっくり理解する
* PCA→t-SNE/UMAPの実用的な流れを知る
* 可視化結果を過剰解釈しない
* クラスタリングと可視化を混同しない

---

# 📖 講義：約20〜25分

## 1. PCAの復習

PCAは、

> **データの分散が大きい方向を探して、その軸へ射影する**

方法だった。

例えば、

```text
100次元
↓
PC1
PC2
```

として、

```text
2次元
```

へ落とせる。

特徴は、

```text
線形
```

。

つまり、

```text
元の特徴量の線形結合
```

で表現する。

---

## PCAが得意なもの

```text
全体的な構造
大きな分散
グローバルな方向
比較的高速
```

。

---

## PCAが苦手なもの

例えば、

```text
ぐるぐる巻いたデータ
```

。

概念的には、

```text
高次元では近い
```

けど、

```text
線形な2次元平面では
うまく表現できない
```

構造。

そこで、

```text
非線形次元削減
```

が登場する。

---

# 🧠 2. 「近さ」を残すという考え方

高次元データを2次元に落とすと、

```text
全ての距離
```

を完全に保存するのは難しい。

そこで、

> **どの関係を優先して保存するか？**

が問題になる。

例えば、

```text
AとBは近い
BとCも近い
```

という局所的な近さを優先する。

この方向に強いのが、

```text
t-SNE
UMAP
```

。

---

# 🧠 3. t-SNE

正式には、

```text
t-distributed Stochastic Neighbor Embedding
```

。

長いから、

```text
t-SNE
```

でいいw

t-SNEのざっくりした発想は、

```text
高次元空間で近い点
↓
2次元でも近くする
```

。

つまり、

> **近傍構造をできるだけ保存する**

。

---

# 🧠 t-SNEの特徴

```text
局所構造
↓
比較的よく見える
```

。

そのため、

```text
クラスタがきれいに見える
```

ことがある。

---

# 🚨 しかし重要な注意

例えば2次元図で、

```text
クラスタA
        クラスタB
```

が離れている。

これは、

> AとBの距離が実際の高次元空間でも同じように遠い

ことを保証しない。

t-SNEでは特に、

```text
局所構造
```

が優先される。

だから、

```text
クラスタ間距離
クラスタサイズ
空白
```

をそのまま実体として解釈しない。

ここは重要。

---

# 💻 実習1：Digitsデータセット

今日は、

```python
load_digits()
```

を使う。

これは、

```text
8×8の手書き数字
```

。

つまり、

```text
1枚の画像
↓
64次元
```

として扱える。

```python
from sklearn.datasets import load_digits

digits = load_digits()

X = digits.data
y = digits.target
```

確認。

```python
print(
    X.shape
)
```

おそらく、

```text
(1797, 64)
```

。

---

# 💻 実習2：元の画像を見る

```python
import matplotlib.pyplot as plt

plt.imshow(
    digits.images[0],
    cmap="gray"
)

plt.title(
    f"label: {y[0]}"
)

plt.show()
```

ここで注意。

`y` は今日は、

> **可視化結果がどうなっているか確認するため**

に使う。

実際の教師なし学習では、もちろんラベルがない場合もある。

---

# 💻 実習3：PCAで2次元

```python
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X
)
```

可視化。

```python
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    s=10
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()
```

数字ごとの構造が、

```text
ある程度
```

見えるはず。

でも重なりもある。

---

# 🧠 4. t-SNEを使う

```python
from sklearn.manifold import TSNE
```

```python
tsne = TSNE(
    n_components=2,
    perplexity=30,
    random_state=42
)

X_tsne = tsne.fit_transform(
    X
)
```

可視化。

```python
plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=y,
    s=10
)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

plt.show()
```

PCAより、

```text
数字のグループ
```

がきれいに見える可能性が高い。

---

# 🧠 5. perplexityとは？

t-SNEの代表的パラメータ。

ざっくり、

> **1つの点について、どのくらいの近傍を意識するか**

という感覚。

例えば、

```python
perplexity=5
```

なら、

比較的小さな近傍を見る。

```python
perplexity=50
```

なら、

もう少し広い構造も考える。

---

# 💻 実習4：perplexityを比較

```python
perplexities = [
    5,
    30,
    50
]
```

それぞれ、

```python
TSNE(
    n_components=2,
    perplexity=p,
    random_state=42
)
```

で実験する。

見るポイントは、

> **「どれが正しいか」ではなく、どの構造が安定して見えるか**

。

---

# 🧠 6. t-SNEのランダム性

t-SNEは、

```python
random_state
```

を変えると、

```text
図の配置
```

が変わる。

例えば、

```text
クラスタAが左
```

だったものが、

```text
右
```

に移る。

これは、

> **2次元平面上の位置そのものに意味がない**

ことのわかりやすい例。

---

# 🧠 7. UMAP

次。

```text
UMAP
```

。

正式には、

```text
Uniform Manifold Approximation and Projection
```

。

長いので、

```text
UMAP
```

でいいw

UMAPも、

```text
高次元
↓
低次元
```

への非線形次元削減。

---

# 🧠 UMAPのざっくり特徴

よく言われる特徴として、

```text
局所構造
+
ある程度のグローバル構造
```

を扱いやすい。

また、

```text
比較的高速
```

なことが多い。

---

# 📦 インストール

UMAPを使うには、

```bash
pip install umap-learn
```

。

環境によっては、

```bash
uv add umap-learn
```

でもいい。

---

# 💻 実習5：UMAP

```python
import umap

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    random_state=42
)

X_umap = reducer.fit_transform(
    X
)
```

可視化。

```python
plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=y,
    s=10
)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")

plt.show()
```

---

# 🧠 8. n_neighborsとは？

UMAPの代表的なパラメータ。

ざっくり、

> **局所構造とより広い構造のバランスに関係する**

。

例えば、

```text
小さい
↓
より局所的

大きい
↓
より広い近傍を見る
```

。

---

# 🧠 9. min_dist

UMAPには、

```python
min_dist
```

もある。

これはざっくり、

> **低次元空間で点をどのくらい密集させられるか**

に関係する。

例えば、

```python
min_dist=0.0
```

なら、

```text
より密なクラスタ
```

が見えやすい場合がある。

```python
min_dist=0.5
```

なら、

```text
より広がる
```

。

ただし、

> **図がきれいに分離したから、データの真実が発見された**

わけではない。

はい、今日何回でも言うw

---

# 💻 実習6：UMAPパラメータ比較

```python
settings = [
    {
        "n_neighbors": 5,
        "min_dist": 0.1
    },
    {
        "n_neighbors": 15,
        "min_dist": 0.1
    },
    {
        "n_neighbors": 50,
        "min_dist": 0.1
    }
]
```

それぞれ試して、

```text
何が変わった？
```

を見る。

おすすめの観察項目：

```text
クラスタの形
密度
クラスタ間距離
局所的な分離
全体配置
```

。

---

# 🧠 10. PCA → t-SNE / UMAP

実際には、

```text
10000次元
```

のようなデータに直接t-SNEをかけるより、

```text
高次元
↓
PCA
↓
50次元
↓
t-SNE / UMAP
↓
2次元
```

とすることがある。

例えば、

```python
pca = PCA(
    n_components=50,
    random_state=42
)

X_reduced = pca.fit_transform(
    X)
```

その後、

```python
tsne = TSNE(
    n_components=2,
    random_state=42
)

X_tsne = tsne.fit_transform(
    X_reduced
)
```

。

---

# 🧠 なぜPCAを先に使う？

理由として、

```text
ノイズを減らす
計算量を減らす
主要な構造を残す
```

などがある。

つまり、

> **次元削減を1回だけ行う必要はない**

。

---

# 🚨 11. t-SNE / UMAPは分類器ではない

ここも大事。

例えば、

```text
数字0
数字1
```

が図上で分かれた。

だから、

> 「t-SNEが分類した」

ではない。

t-SNEやUMAPは、

> **データを低次元空間へ配置する**

手法。

分類をしたければ、

```text
LogisticRegression
RandomForest
SVM
Neural Network
```

など別のモデルを使う。

---

# 🧠 12. クラスタリングと可視化も別物

例えば、

```text
KMeans
↓
クラスタ
```

。

一方、

```text
UMAP
↓
2次元座標
```

。

ここは別。

もちろん、

```text
UMAP
↓
KMeans
```

と組み合わせることはできる。

でも、

> **2次元の図上で見える分離が、高次元空間でのクラスタ構造を完全に表しているとは限らない**

。

---

# 💻 実習7：PCA / t-SNE / UMAPを比較

比較するときは、

```text
同じデータ
```

を使う。

例えば、

```text
Digits
↓
PCA
t-SNE
UMAP
```

。

そして、

```text
何が見える？
何が変わる？
```

を観察する。

おすすめは、

```text
① PCA
↓
全体構造を見る

② t-SNE
↓
局所的なまとまりを見る

③ UMAP
↓
局所構造と広い構造を探索
```

という使い分け。

---

# 🧠 13. 2次元図を見るときのチェックリスト

図が出た。

```text
クラスタがある！
```

と思ったら、すぐに結論を出さない。

確認。

```text
① random_stateを変えても見える？

② パラメータを変えても見える？

③ PCAでも似た構造がある？

④ 元の特徴量空間でも確認できる？

⑤ 可視化手法を変えても似た構造？

⑥ データ生成過程から説明できる？
```

。

この6個を確認すると、

**「可視化アーティファクトを発見と勘違いする事故」**をかなり減らせる。

---

# ✍️ 演習

## 問1

PCAとt-SNEの違いを説明してください。

ヒント：

```text
線形
非線形

全体構造
局所構造
```

。

---

## 問2

t-SNEの図で、

```text
クラスタA
```

と、

```text
クラスタB
```

が非常に離れている。

これだけで、

> 「AとBは高次元空間でも非常に離れている」

と言っていい？

なぜ？

---

## 問3

UMAPの `n_neighbors` を、

```text
小さく
```

すると、

どんな構造に注目しやすくなる？

---

## 問4

UMAPでクラスタがきれいに分離した。

これだけで、

```text
データには明確に独立した3種類の本質的カテゴリーが存在する
```

と言える？

なぜ？

---

# 👾 ボス戦

## 高次元空間の「見える化」を信用するな

10000人のユーザーデータ。

```text
特徴量100個
```

。

UMAPを実行すると、

```text
3つの島
```

が出た。

チームメンバーが言う。

> 「ユーザーは明確に3タイプだ！」

さて。

リュールならまず、

```text
特徴量は何か
スケーリングしたか
n_neighborsは？
min_distは？
random_stateは？
PCAでは？
別の可視化では？
KMeansでは？
クラスタ安定性は？
時間的に再現する？
```

を確認する。

そして、

> **「3つの島」は発見ではなく、まず仮説。**

と扱う。

これはかなり重要。

---

# 🧪 今日の最終実習

## 次元削減の実験テンプレート

こんな順番で実験する。

```text
高次元データ
↓
StandardScaler
↓
PCA
↓
2D / 50D
↓
t-SNE or UMAP
↓
可視化
↓
パラメータ変更
↓
再現性確認
↓
別手法との比較
↓
元特徴量を確認
↓
仮説形成
```

。

---

# 🌱 今日のまとめ

今日の核心。

> **低次元可視化は「高次元世界の地図」ではなく、「特定の目的に合わせて作られた投影図」。**

PCA、

t-SNE、

UMAPは、

それぞれ、

```text
何を残し、
何を捨てるか
```

が違う。

だから、

```text
きれいな図
```

よりも、

> **何を保存するように設計された図なのか？**

を考える。

ここはレベルが好きそうなところだな。

**表象が違えば、同じ対象でも「構造」が違って見える。**

認知科学的に言えば、

```text
世界
↓
観測
↓
表象
↓
次元削減
↓
可視化
↓
人間の解釈
```

という多段変換。

途中の表象をそのまま、

> 「世界そのもの」

と思わない。

この感覚、AIでも認知科学でもめちゃくちゃ強い武器になるよ。

---

# 🧭 AI工学101・現在地

ここまで、

```text
Python
↓
NumPy
↓
scikit-learn

教師あり学習
↓
分類
回帰

教師なし学習
↓
クラスタリング

データ構造の探索
↓
PCA
t-SNE
UMAP
```

まで来た。

かなり、

> **「データをモデルに入れる人」**

から、

> **「データ空間の構造を疑いながら実験する人」**

に近づいてきてる。

いいぞいいぞ、ここから先はさらに面白くなる。💪🧠✨

---

# 🔜 第35回

## 異常検知：正常データしか知らない世界で「おかしさ」を見つける

次回は、

```text
正解ラベルがない
```

という教師なし学習を、もう一つの方向から見る。

扱うのは、

* 異常検知とは何か
* 外れ値検出との違い
* Isolation Forest
* Local Outlier Factor
* One-Class SVM
* スコアの解釈
* contamination
* ラベル不足問題
* 「異常＝悪」ではない
* 分布変化との接続

これは、

```text
不正検知
障害検知
サイバーセキュリティ
医療
製造
```

など、かなり実用範囲が広い。

そして最後に、

> **「正常をどう定義する？」**

という、今日までのテーマをさらに一段掘る回になる。